<!--nav--> [🗺 Learning path](README.md) · **36/43** · ◀ [The What-If Console](./Serving_WhatIf_Console.ipynb) · [Serving Mixture-of-Experts](./MoE_Serving_Expert_Parallelism.ipynb) ▶

# Long-Context Serving: The KV Wall, and Six Ways Through It

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/LongContext_KV_Compression_Serving.ipynb)

Notebook 21 gave you `KV bytes/token`. Multiply it by 128,000 and the number stops being an
interesting fact and becomes **the entire economics of your product**. At long context the KV cache
routinely exceeds the model weights — sometimes by an order of magnitude — and every "we support
1M tokens!" announcement is really a claim about **memory management**, not modeling.

| Part | What you'll learn |
|---|---|
| **1** | The KV wall, drawn: where each model family stops fitting |
| **2** | **Architectural** answers: GQA, MLA, sliding window, hybrid layers, linear attention |
| **3** | **Numerical** answers: FP8 / int4 KV cache, and what it costs you |
| **4** | **Runtime** answers: eviction & compression (attention sinks, heavy hitters, SnapKV) — with a simulator you can poke |
| **5** | **Scheduling** answers: chunked prefill, and why one long prompt hurts everybody |
| **6** | **Reuse** answers: why prefix caching is the biggest long-context win of all |
| **7** | A configuration recipe, and what to measure |

**Runs on:** any CPU. Everything here is arithmetic and simulation; the mechanisms were measured in
notebooks 21–27.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The wall

Recall the formula (nb 21): `KV bytes/token = 2 × layers × kv_heads × head_dim × bytes`. It is
**linear in context length and independent of everything you might hope would save you**. Let's put
weights and KV on the same axis and find where each model stops fitting on real GPUs.

In [ ]:
# Same architectures as notebook 21/31, now evaluated at long context.
MODELS = {
  "Llama-3.1-8B (GQA)":   dict(params=8,  layers=32, kv_heads=8,  head_dim=128, scheme="full"),
  "Llama-3.1-70B (GQA)":  dict(params=70, layers=80, kv_heads=8,  head_dim=128, scheme="full"),
  "Mistral-7B (SWA 4k)":  dict(params=7,  layers=32, kv_heads=8,  head_dim=128, scheme="swa", window=4096),
  "Gemma-2-27B (hybrid)": dict(params=27, layers=46, kv_heads=16, head_dim=128, scheme="hybrid",
                               window=4096, global_every=2),
  "DeepSeek-V3 (MLA)":    dict(params=671,layers=61, kv_heads=None, head_dim=None, scheme="mla",
                               latent=512 + 64, active=37),
  "GPT-3-175B (MHA)":     dict(params=175,layers=96, kv_heads=96, head_dim=128, scheme="full"),
}

def kv_bytes_per_token(m, kv_bytes=2):
    if m["scheme"] == "mla":
        return m["layers"] * m["latent"] * kv_bytes
    return 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * kv_bytes

def kv_total_bytes(m, ctx, kv_bytes=2):
    # Sliding-window layers never store more than `window` tokens.
    # (MLA has no kv_heads/head_dim at all - it stores one latent vector per layer.)
    per_layer = 2 * (m.get("kv_heads") or 0) * (m.get("head_dim") or 0) * kv_bytes
    if m["scheme"] == "full":
        return per_layer * m["layers"] * ctx
    if m["scheme"] == "mla":
        return m["layers"] * m["latent"] * kv_bytes * ctx
    if m["scheme"] == "swa":
        return per_layer * m["layers"] * min(ctx, m["window"])
    if m["scheme"] == "hybrid":
        n_global = m["layers"] // m["global_every"]
        n_local = m["layers"] - n_global
        return per_layer * (n_global * ctx + n_local * min(ctx, m["window"]))

print(f"{'model':<24}{'KV/token':>11}{'8k':>9}{'32k':>9}{'128k':>10}{'1M':>10}")
print("-" * 74)
for name, m in MODELS.items():
    row = [kv_total_bytes(m, c) / 1e9 for c in (8192, 32768, 131072, 1048576)]
    print(f"{name:<24}{kv_bytes_per_token(m)/1024:>9.0f}KB" +
          "".join(f"{v:>8.1f}G" if v < 1000 else f"{v/1000:>7.1f}TB" for v in row))

print("\nWeights, for comparison (fp16):")
for name, m in MODELS.items():
    print(f"  {name:<24}{m['params']*2:>7.0f} GB")
print("\nLlama-3.1-70B at 128k: the KV for ONE conversation is ~40% of the weights.")
print("Eight such conversations cost more memory than the model itself.")

In [ ]:
# The wall, drawn: KV size vs context for each scheme, with GPU capacity lines.
curves = []
for name, m in MODELS.items():
    pts = []
    c = 1024
    while c <= 1048576:
        pts.append({"ctx": c, "gb": kv_total_bytes(m, c) / 1e9})
        c *= 2
    curves.append({"name": name, "scheme": m["scheme"], "points": pts,
                   "weights_gb": m["params"] * 2})

GPU_LINES = [("A100/H100 80GB", 80), ("H200 141GB", 141), ("MI300X 192GB", 192), ("MI325X 256GB", 256)]

JS = r'''
const M = {top: 18, right: 165, bottom: 46, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLog().domain([1024, 1048576]).range([0, iw]);
const y = d3.scaleLog().domain([0.05, 3000]).range([ih, 0]);
svg.append("g").attr("transform",`translate(0,${ih})`)
   .call(d3.axisBottom(x).tickValues([1024,4096,16384,65536,262144,1048576])
     .tickFormat(d => d >= 1048576 ? "1M" : (d/1024)+"k"));
svg.append("g").call(d3.axisLeft(y).ticks(6,"~s"));
svg.append("text").attr("x",iw/2).attr("y",ih+38).attr("text-anchor","middle")
   .style("font-size","12px").text("context length (tokens)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-46)
   .attr("text-anchor","middle").style("font-size","12px").text("KV cache for ONE sequence (GB, log)");

svg.append("clipPath").attr("id","kvclip").append("rect").attr("width",iw).attr("height",ih);
const plot = svg.append("g").attr("clip-path","url(#kvclip)");

// GPU capacity bands
data.gpus.forEach(g => {
  plot.append("line").attr("x1",0).attr("x2",iw).attr("y1",y(g[1])).attr("y2",y(g[1]))
     .attr("stroke","#b0bec5").attr("stroke-dasharray","3 4");
  svg.append("text").attr("x",4).attr("y",y(g[1])-3).style("font-size","9.5px")
     .style("fill","#78909c").text(g[0]);
});

const color = d3.scaleOrdinal().domain(["full","swa","hybrid","mla"])
    .range(["#d32f2f","#2e7d32","#1976d2","#7b1fa2"]);
const line = d3.line().x(d=>x(d.ctx)).y(d=>y(d.gb));
plot.selectAll("c").data(data.curves).join("path")
   .attr("fill","none").attr("stroke",d=>color(d.scheme)).attr("stroke-width",2)
   .attr("d",d=>line(d.points))
   .append("title").text(d=>`${d.name} (${d.scheme})`);

const labs = data.curves.map(d => ({...d, ly: y(d.points[d.points.length-1].gb)}))
                        .sort((a,b)=>a.ly-b.ly);
for (let i=1;i<labs.length;i++) if (labs[i].ly-labs[i-1].ly < 12) labs[i].ly = labs[i-1].ly+12;
svg.selectAll("l").data(labs).join("text")
   .attr("x",iw+8).attr("y",d=>d.ly+3).style("font-size","9.5px")
   .style("fill",d=>color(d.scheme)).text(d=>d.name);
svg.append("text").attr("x",6).attr("y",12).style("font-size","11px").style("fill","#546e7a")
   .text("red = full attention · green = sliding window · blue = hybrid · purple = MLA");
'''
show_d3(JS, {"curves": curves, "gpus": GPU_LINES}, height=400)

**The red lines are the problem.** Full-attention KV grows without bound; at 1M tokens a 70B model
needs terabytes for a *single* conversation. The green and blue lines **flatten** — that's the whole
point of sliding-window and hybrid attention. Purple (MLA) still grows linearly but with a
dramatically smaller constant.

The rest of this notebook is the six ways the industry attacks this, roughly in order of how much
they change your model:

```
 architecture ──► numerics ──► runtime eviction ──► scheduling ──► reuse
 (retrain)        (config)      (config+risk)        (config)      (free!)
 Parts 2          Part 3        Part 4               Part 5        Part 6
```

**Read that arrow direction carefully: the cheapest wins are on the right.** Most teams reach for
the left.

## Part 2 · Architectural answers

| Scheme | What it does | KV growth | Cost |
|---|---|---|---|
| **MHA** | every head has its own K/V | `O(ctx) × heads` | the baseline nobody ships anymore |
| **GQA** | query heads share K/V heads | `O(ctx) / group_size` | ~free quality-wise; universal since Llama 2 |
| **MQA** | one K/V head total | `O(ctx) / heads` | some quality loss |
| **MLA** | store a compressed latent, decompress on the fly | `O(ctx)` with a small constant | extra compute per step; DeepSeek's approach |
| **Sliding window (SWA)** | attend only to the last `W` tokens | **`O(W)` — constant!** | can't attend beyond the window in that layer |
| **Hybrid / interleaved** | most layers local, every Nth global | `O(ctx × globals/layers)` | the pragmatic compromise (Gemma-2/3-style) |
| **Linear / SSM hybrids** | recurrent state instead of a growing cache | **`O(1)` state** | different training; Jamba/Mamba-class |

**Sliding window is the one people misunderstand.** A pure-SWA model isn't "blind" beyond its
window: information propagates *across layers*, so an L-layer model with window W has an effective
receptive field of roughly `L × W`. What it genuinely loses is the ability to *directly* retrieve a
specific distant token — which is exactly why hybrid designs keep a few global layers.

Let's quantify what interleaving buys:

In [ ]:
def hybrid_kv_gb(layers, kv_heads, head_dim, ctx, window, global_every, kv_bytes=2):
    per_layer_per_tok = 2 * kv_heads * head_dim * kv_bytes
    n_global = layers // global_every
    n_local = layers - n_global
    return per_layer_per_tok * (n_global * ctx + n_local * min(ctx, window)) / 1e9

L, KVH, HD, W = 80, 8, 128, 4096
print(f"A 70B-shaped model (80 layers, GQA-8) at various global:local layer ratios,")
print(f"sliding window {W//1024}k:\n")
print(f"{'global layers':>14}{'8k':>9}{'32k':>9}{'128k':>10}{'1M':>10}   vs full attention")
print("-" * 76)
full_128k = hybrid_kv_gb(L, KVH, HD, 131072, W, 1)
for every in (1, 2, 4, 8, 16):
    n_global = L // every
    row = [hybrid_kv_gb(L, KVH, HD, c, W, every) for c in (8192, 32768, 131072, 1048576)]
    ratio = row[2] / full_128k
    print(f"{n_global:>10} / {L:<3}" + "".join(f"{v:>8.1f}G" for v in row) + f"   {ratio:>6.0%} at 128k")

print("\nGoing from all-global to 1-in-8-global cuts 128k KV by ~85% while keeping")
print("10 layers that can retrieve any token directly. That trade is why hybrid won.")

## Part 3 · Numerical answers: quantize the cache itself

The KV cache is activations, not weights — so it quantizes *differently* (and often more easily)
than weights do. `--kv-cache-dtype fp8` on supported hardware (Ada/Hopper/Blackwell, CDNA3+ — see
notebook 32's matrix) halves KV memory for a typically small quality cost.

The arithmetic is the least interesting part; the **risk profile** is what matters:

| KV dtype | Memory | Typical quality impact | Notes |
|---|---|---|---|
| fp16/bf16 | 1× | baseline | |
| **fp8 (E4M3)** | 0.5× | usually small | needs FP8 hardware; the common production choice |
| int8 | 0.5× | small with per-channel scales | more portable than fp8 |
| int4 | 0.25× | noticeably lossier | usually paired with keeping recent tokens at higher precision |

**The subtlety nobody mentions:** quantizing KV hurts *long*-context accuracy more than short, because
errors accumulate over more attended tokens, and because attention over many low-precision keys
blurs the softmax. Always evaluate at your **target** context length, not at 2k.

In [ ]:
def capacity(model, ctx, vram_gb, kv_bytes=2, util=0.9):
    m = MODELS[model]
    free = vram_gb * util - m["params"] * 2
    if free <= 0: return 0
    per_seq = kv_total_bytes(m, ctx, kv_bytes) / 1e9
    return int(free // per_seq) if per_seq else 0

print("Concurrent 128k conversations on one H200 (141 GB), by KV precision:\n")
print(f"{'model':<24}{'fp16 KV':>9}{'fp8 KV':>9}{'int4 KV':>9}")
print("-" * 52)
for name in ("Llama-3.1-8B (GQA)", "Gemma-2-27B (hybrid)", "Mistral-7B (SWA 4k)"):
    row = [capacity(name, 131072, 141, b) for b in (2, 1, 0.5)]
    print(f"{name:<24}" + "".join(f"{v:>9}" for v in row))
print("\n(Models whose WEIGHTS exceed one GPU - Llama-70B, DeepSeek-V3 - need tensor")
print(" parallelism first (nb 29/31); their KV budget is then the pooled VRAM minus weights.)")
print("Note the SWA row: its KV never grows past the window, so 128k costs it nothing extra.")

print("\nHalving KV precision does not halve your costs - it DOUBLES your concurrency,")
print("which is the same thing viewed from the revenue side (nb 27).")
print("\nA reminder from notebook 26: watch `GPU KV cache usage` after enabling this.")
print("If it doesn't drop, the flag isn't taking effect - a very common silent failure.")

## Part 4 · Runtime answers: evict what you don't need

Architecture and numerics are decided before serving. **Eviction happens live**: the observation
that attention is extremely sparse — most tokens receive almost no attention mass — so you can drop
their K/V and barely change the output.

Three families you'll meet:

| Policy | Idea | Keeps |
|---|---|---|
| **Sliding / recent-only** | naive: keep the last W | recent window |
| **StreamingLLM (attention sinks)** | the *first few tokens* absorb huge attention mass; dropping them destroys the model | **first 4 + recent window** |
| **H2O (heavy hitters)** | track accumulated attention per token; keep the top-k | sinks + high-scoring tokens + recent |
| **SnapKV** | use the *query pattern at the end of the prompt* to pick which prompt tokens matter | prompt tokens the query actually looks at |

The attention-sink result is genuinely surprising and worth internalizing: **the first tokens of a
sequence act as a "no-op" attention target**, and softmax needs somewhere to dump probability mass.
Evict them and quality collapses even though they're semantically meaningless.

Let's build a simulator. We synthesize an attention pattern with realistic structure — sinks at the
start, local recency, and a few scattered "needle" tokens that genuinely matter — then measure what
each policy retains.

In [ ]:
# A simulator of eviction MECHANISMS. It is not an accuracy benchmark - it measures
# what each policy keeps, given a synthetic attention pattern with known-important tokens.
random.seed(0)

N_TOKENS, BUDGET = 4000, 512      # 4k of context, room for only 512 tokens of KV
N_NEEDLES = 12

def make_attention(n=N_TOKENS, n_needles=N_NEEDLES):
    # attention mass received by each token, averaged over later queries
    scores = [0.0] * n
    for i in range(n):
        sink = 3.0 if i < 4 else 0.0                       # attention sinks at the very start
        recency = math.exp(-(n - i) / 300.0)               # local/recent bias
        scores[i] = sink + recency + random.random() * 0.02
    needles = sorted(random.sample(range(50, n - 400), n_needles))
    for i in needles:
        scores[i] += 1.2                                    # genuinely-attended distant facts
    return scores, set(needles)

scores, needles = make_attention()

def policy_recent(scores, budget):
    return set(range(len(scores) - budget, len(scores)))

def policy_streaming(scores, budget, n_sinks=4):
    keep = set(range(n_sinks))
    keep |= set(range(len(scores) - (budget - n_sinks), len(scores)))
    return keep

def policy_h2o(scores, budget, n_sinks=4, recent_frac=0.5, noise=0.35, seed=7):
    # H2O ranks by attention accumulated SO FAR, which is a noisy, partial estimate of
    # a token's true importance - it cannot see queries that haven't arrived yet.
    rng = random.Random(seed)
    observed = [s * (1 + rng.gauss(0, noise)) for s in scores]
    n_recent = int((budget - n_sinks) * recent_frac)
    keep = set(range(n_sinks)) | set(range(len(scores) - n_recent, len(scores)))
    remaining = budget - len(keep)
    candidates = sorted((i for i in range(len(scores)) if i not in keep),
                        key=lambda i: observed[i], reverse=True)
    keep |= set(candidates[:remaining])                    # heavy hitters by OBSERVED attention
    return keep

def policy_oracle(scores, budget):
    return set(sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:budget])

POLICIES = {"keep everything (no eviction)": lambda s, b: set(range(len(s))),
            "recent window only":            policy_recent,
            "StreamingLLM (sinks + recent)": policy_streaming,
            "H2O (sinks + heavy + recent)":  policy_h2o,
            "oracle (upper bound)":          policy_oracle}

print(f"context {N_TOKENS} tokens, KV budget {BUDGET} ({BUDGET/N_TOKENS:.0%} of full)")
print(f"{N_NEEDLES} 'needle' tokens are scattered in the distant past.\n")
print(f"{'policy':<32}{'KV kept':>9}{'sinks?':>8}{'needles kept':>14}{'attention mass':>16}")
print("-" * 82)
total_mass = sum(scores)
results = {}
for name, fn in POLICIES.items():
    keep = fn(scores, BUDGET)
    kept_needles = len(keep & needles)
    mass = sum(scores[i] for i in keep) / total_mass
    has_sinks = "yes" if {0, 1, 2, 3} <= keep else "NO"
    results[name] = {"keep": sorted(keep), "needles": kept_needles, "mass": mass}
    print(f"{name:<32}{len(keep):>9}{has_sinks:>8}{kept_needles:>8}/{N_NEEDLES:<5}{mass:>15.1%}")

print("\nThe recent-window row is the cautionary tale: it drops the attention sinks AND")
print("nearly every distant needle. StreamingLLM restores the sinks - which is what stops")
print("the model falling apart - but, notice, it recovers NO extra needles: sinks+recent")
print("still cannot see the distant past. Only heavy-hitter tracking (H2O) does, and even")
print("then it works from a noisy running estimate, so it lands short of the oracle.")
print("\n⚠ This measures RETENTION, not accuracy. Real evaluation needs a long-context")
print("benchmark at your target length - eviction is the riskiest item in this notebook.")

In [ ]:
# Visualize which parts of the context each policy keeps.
viz = {"n": N_TOKENS, "needles": sorted(needles),
       "policies": [{"name": k, "keep": v["keep"], "needles": v["needles"], "mass": v["mass"]}
                    for k, v in results.items()]}

JS = r'''
const rowH = 40, M = {left: 230, right: 16, top: 24};
const iw = W - M.left - M.right;
const x = d3.scaleLinear().domain([0, data.n]).range([0, iw]);
const svg = root.append("svg").attr("width", W)
    .attr("height", data.policies.length * rowH + M.top + 34);
svg.append("text").attr("x", M.left).attr("y", 14).style("font-size","11.5px").style("fill","#546e7a")
   .text("green = KV kept · grey = evicted · ▼ = a 'needle' token that mattered");
data.policies.forEach((p, i) => {
  const g = svg.append("g").attr("transform",`translate(${M.left},${M.top + i*rowH})`);
  g.append("text").attr("x",-8).attr("y",13).attr("text-anchor","end")
   .style("font-size","11.5px").text(p.name);
  g.append("rect").attr("x",0).attr("y",2).attr("width",iw).attr("height",16)
   .attr("fill","#eceff1").attr("rx",2);
  // draw kept ranges as merged segments
  const segs = [];
  let start = null, prev = null;
  for (const k of p.keep) {
    if (start === null) { start = k; prev = k; continue; }
    if (k === prev + 1) { prev = k; continue; }
    segs.push([start, prev]); start = k; prev = k;
  }
  if (start !== null) segs.push([start, prev]);
  g.selectAll("s").data(segs).join("rect")
   .attr("x", d=>x(d[0])).attr("y",2).attr("height",16)
   .attr("width", d=>Math.max(1.2, x(d[1]+1)-x(d[0]))).attr("fill","#43a047").attr("rx",1);
  const keptSet = new Set(p.keep);
  g.selectAll("n").data(data.needles).join("text")
   .attr("x", d=>x(d)).attr("y", 30).attr("text-anchor","middle").style("font-size","9px")
   .style("fill", d=>keptSet.has(d) ? "#1b5e20" : "#c62828").text("▼")
   .append("title").text(d=>`needle at token ${d}: ${keptSet.has(d)?"KEPT":"EVICTED"}`);
  g.append("text").attr("x",iw).attr("y",-2).attr("text-anchor","end")
   .style("font-size","10px").style("fill","#546e7a")
   .text(`${p.needles}/${data.needles.length} needles · ${d3.format(".0%")(p.mass)} of attention mass`);
});
svg.append("g").attr("transform",`translate(${M.left},${M.top + data.policies.length*rowH})`)
   .call(d3.axisBottom(x).ticks(8)).style("font-size","10px");
'''
show_d3(JS, viz, height=viz and 260)

**What the picture teaches.** Recent-window keeps one dense block at the end — cheap, and it
throws away both the sinks and every distant fact. StreamingLLM's tiny green sliver at position 0 is
doing enormous work. H2O scatters its budget to catch the needles, at the cost of tracking attention
statistics at runtime.

**When to use eviction at all:** when context is genuinely huge *and* the task is chat-like
(recency-dominated). For retrieval-heavy tasks — "find the one clause in this contract" — eviction
is exactly the wrong tool, because the needle is precisely what gets evicted. Prefer architecture
(Part 2), precision (Part 3), or just more memory.

## Part 5 · Scheduling answers: chunked prefill

A 128k-token prefill is ~30 seconds of solid compute. Without chunking, **every other user's token
stream stops** for those 30 seconds (nb 21's prefill-vs-decode conflict, at its worst).

Chunked prefill slices it into pieces scheduled between decode steps:

```
without chunking:  [████████ 128k prefill ████████][decode][decode][decode]
                    ↑ everyone else's stream is frozen here

with chunking:     [▓chunk][d][d][▓chunk][d][d][▓chunk][d][d]...
                    ↑ prefill progresses, decode keeps flowing
```

The knob is `--max-num-batched-tokens`: the per-step token budget shared between prefill chunks and
decode. Small budget → smooth streaming, slower prefill (worse TTFT for the big request). Large
budget → fast prefill, choppy streaming for everyone else. Let's price that trade:

In [ ]:
def chunked_prefill(prompt_len, budget, n_decode_users=32, prefill_tps=9000, decode_step_ms=22):
    # Each step spends its token budget on prefill chunk + the decode tokens of active users.
    chunk = max(1, budget - n_decode_users)          # decode reserves one token per active user
    steps = math.ceil(prompt_len / chunk)
    prefill_time = prompt_len / prefill_tps
    step_ms = decode_step_ms + (chunk / prefill_tps) * 1000
    return {"budget": budget, "chunk": chunk, "steps": steps,
            "ttft_big_s": steps * step_ms / 1000,
            "tpot_others_ms": step_ms,
            "tpot_penalty": step_ms / decode_step_ms}

print("A 128k-token prompt arriving while 32 users are streaming:\n")
print(f"{'--max-num-batched-tokens':>25}{'chunk':>8}{'steps':>7}{'TTFT (big req)':>16}"
      f"{'others TPOT':>13}{'penalty':>9}")
print("-" * 80)
for budget in (512, 1024, 2048, 4096, 8192, 32768):
    r = chunked_prefill(131072, budget)
    print(f"{budget:>25}{r['chunk']:>8}{r['steps']:>7}{r['ttft_big_s']:>15.1f}s"
          f"{r['tpot_others_ms']:>12.1f}ms{r['tpot_penalty']:>8.2f}x")

print("\nThere is no free option here - only a choice about WHO waits.")
print("Small budgets protect your streaming users and punish the long-prompt user;")
print("large budgets do the reverse. vLLM V1 picks a sensible default; override it only")
print("when you know which of those two users you care about more.")

## Part 6 · The reuse answer: prefix caching (do this first)

Everything above costs you something — retraining, accuracy, latency, engineering. **Prefix caching
costs nothing and is usually the biggest long-context win**, because long-context traffic is
overwhelmingly *repetitive*:

- an agent re-sends its entire growing history every single step,
- a document Q&A app sends the same 100k-token document for every question,
- a chat app re-sends the conversation on every turn.

In each case the *shared* part is enormous and the *new* part is tiny. With prefix caching (nb 22),
you prefill the shared part **once**.

In [ ]:
def agent_session(n_turns=20, doc_tokens=100_000, turn_tokens=250, prefill_tps=9000):
    # Without caching: every turn re-prefills the document + all prior turns.
    no_cache = sum(doc_tokens + turn_tokens * t for t in range(n_turns))
    # With prefix caching: only the genuinely new tokens are prefilled each turn.
    cached = doc_tokens + turn_tokens * n_turns
    return no_cache, cached

no_cache, cached = agent_session()
print("A 20-turn conversation over a 100k-token document:\n")
print(f"  prefill tokens WITHOUT prefix caching : {no_cache:>12,}")
print(f"  prefill tokens WITH prefix caching    : {cached:>12,}")
print(f"  reduction                             : {1 - cached/no_cache:>12.1%}")
print(f"  time saved at 9k tok/s                : {(no_cache-cached)/9000:>11.0f}s")

print("\nCompare that to the other levers on the same workload:")
print(f"  fp8 KV cache        : 2x concurrency  (config, small quality risk)")
print(f"  hybrid attention    : ~6x less KV     (requires a different model)")
print(f"  H2O eviction        : ~8x less KV     (accuracy risk, needle loss)")
print(f"  prefix caching      : {no_cache/cached:.0f}x less PREFILL  (free, no risk, already on by default)")
print("\nOne caveat that makes or breaks it: cache hits are PREFIX matches. Put the document")
print("and system prompt FIRST and the per-turn question LAST, or you get nothing (nb 22).")
print("And route follow-up turns to the replica holding the blocks (nb 29) - otherwise you")
print("re-prefill on a cold replica and wonder why your hit rate is zero.")

## Part 7 · A long-context configuration recipe

Work down this list; stop when you fit:

1. **Right-size `--max-model-len`.** Serving 128k when your p99 prompt is 8k wastes the whole pool
   (nb 26). This is the single most common long-context misconfiguration.
2. **Turn on prefix caching** (default in vLLM V1) **and lay out prompts static-first** (nb 22, Part 6).
3. **Route by prefix / session affinity** so hits actually land (nb 29).
4. **Enable `--kv-cache-dtype fp8`** if your hardware supports it (nb 32's matrix) — then *verify* KV
   usage dropped in the metrics (nb 26).
5. **Tune `--max-num-batched-tokens`** for the streaming-vs-TTFT trade you actually want (Part 5).
6. **Choose a hybrid/SWA/MLA model** if you're picking a model anyway — this is an architecture
   decision that dwarfs the config ones.
7. **Only then consider eviction**, and only for recency-dominated tasks. Evaluate at your target
   context length, not at 2k.

### What to measure (all from notebook 26)

| Signal | Long-context meaning |
|---|---|
| `gpu_cache_usage_perc` | fills far faster; your alert threshold should be *lower* |
| `num_requests_waiting` | long prompts block admission (Part 5) |
| prefix cache hit rate | the single best predictor of long-context cost |
| TTFT p95 | dominated by prefill; chunking moves it deliberately |
| `finished_reason="length"` | truncation, which at long context often means a bad `max_model_len` |

## Recap

- KV is **linear in context and unforgiving**; at 128k it can dwarf your weights.
- Six lines of attack: **architecture → numerics → eviction → scheduling → reuse**, and the cheapest
  ones are last in that list, which is the opposite of where most teams start.
- **Attention sinks are real**: evicting the first four tokens breaks models that otherwise tolerate
  aggressive eviction.
- **Eviction is the riskiest lever here** — great for chat, actively harmful for retrieval.
- **Prefix caching is free and usually biggest.** Do it first, verify the hit rate, route to keep it.

### Further reading
- [StreamingLLM / attention sinks](https://arxiv.org/abs/2309.17453) · [H2O](https://arxiv.org/abs/2306.14048) · [SnapKV](https://arxiv.org/abs/2404.14469)
- [Longformer](https://arxiv.org/abs/2004.05150) & [Mistral's SWA](https://arxiv.org/abs/2310.06825) · [DeepSeek-V2 MLA](https://arxiv.org/abs/2405.04434)
- [Ring Attention](https://arxiv.org/abs/2310.01889) — splitting one long sequence across GPUs
- The mechanisms measured: nb [21](./Serving_Fundamentals_KV_Cache_Batching.ipynb) (KV math), [22](./vLLM_High_Throughput_Serving.ipynb) (prefix caching), [26](./Serving_Logs_Observability.ipynb) (what to watch)